In [ ]:
# Import required libraries
import pandas as pd                    # Data manipulation and analysis
from rapidfuzz import process, fuzz    # Fuzzy string matching library
import re                               # Regular expressions for text processing

In [ ]:
# Define input and output file paths
# Input: Excel file with transaction and ledger data
# Output: To be assigned later after categorization
input = "credit_txn_v5.xlsx"
output = "output_file.xlsx"

In [ ]:
# Load transaction data from Excel file
df = pd.read_excel(input)

In [ ]:
# Normalize ledger names for consistent comparison
# This function standardizes ledger names by:
#   1. Converting to lowercase for case-insensitive matching
#   2. Removing special characters (keeping only letters and spaces)
#   3. Replacing 3+ consecutive duplicate characters with single character (e.g., "charrrge" → "charge")
#   4. Collapsing multiple spaces into single spaces and trimming

def normalize_ledger(name):
    name = str(name).lower()                                       # Convert to lowercase
    name = re.sub(r'[^a-z\s]', ' ', name)                          # Replace non-letter/space chars with space
    name = re.sub(r'(.)\1{2,}', r'\1', name)                       # Replace 3+ duplicate chars with single char
    name = re.sub(r'\s+', ' ', name).strip()                       # Collapse spaces and trim
    return name

# Apply normalization to create a new column with normalized ledger names
df['ledger_norm'] = df['Ledger Name'].apply(normalize_ledger)

In [ ]:
# Calculate frequency of normalized ledger names
# This identifies how often each normalized ledger appears in the dataset
# Sorted in descending order to prioritize high-frequency ledgers

freq_df = (
    df.groupby('ledger_norm')
      .size()                                                       # Count occurrences of each normalized ledger
      .reset_index(name='frequency')                                # Convert to DataFrame with frequency column
      .sort_values('frequency', ascending=False)                    # Sort by frequency (highest first)
      .reset_index(drop=True)                                       # Reset index after sorting
)

In [ ]:
# Extract list of unique normalized ledger names for fuzzy matching
unique_ledgers = freq_df['ledger_norm'].values.tolist()

In [ ]:
# Filter out low-frequency ledgers from categorization
# Ledgers with fewer than 5 transactions are excluded from fuzzy matching
# This improves accuracy by focusing on significant ledgers
# Creates a category list for matching and frequency lookup

MIN_FREQ = 5

category_df = freq_df[freq_df['frequency'] >= MIN_FREQ].copy()     # Filter high-frequency ledgers
category_list = category_df['ledger_norm'].tolist()                # Extract ledger names as list
category_freq = dict(zip(category_df['ledger_norm'], category_df['frequency']))  # Create lookup dict for frequencies

In [ ]:
# Use RapidFuzz library to find similar normalized ledger names
# This clusters ledgers based on string similarity using token_sort_ratio algorithm
# Ledgers similar by 90%+ are grouped together under the highest frequency ledger in that group
# This helps identify ledgers that are essentially the same despite naming variations

SIM_THRESHOLD = 90  # Similarity threshold in percentage

# Function to find the best matching category for a given ledger
def match_category(ledger):
    match = process.extractOne(
        ledger,
        category_list,
        scorer=fuzz.token_sort_ratio,                              # Token sort ratio is more flexible with word order
        score_cutoff=SIM_THRESHOLD                                 # Only accept matches above 90% similarity
    )
    return match[0] if match else "OTHER"

# Build a mapping of each unique ledger to its matched category
ledger_to_category = {}

for i, ledger in enumerate(unique_ledgers):
    ledger_to_category[ledger] = match_category(ledger)

    # Print progress indicator every 1000 ledgers to track processing
    if i % 1000 == 0:
        print(f"Processed {i}/{len(unique_ledgers)}")

# Add the ledger category to the main dataframe
df['Ledger Category'] = df['ledger_norm'].map(ledger_to_category)

In [ ]:
# End of fuzzy matching categorization
# Data is now enhanced with Ledger Category column based on RapidFuzz similarity matching